# Chandra OCR 2 — 이미지 텍스트 추출 서버
- 모델: `datalab-to/chandra-ocr-2`
- 권장 환경: RTX 4070 (VRAM 8GB+), Python 3.12

In [2]:
# # ──────────────────────────────────────────────────────────────
# # 패키지 설치 (최초 1회)
# # torch 2.7.0+cu118이 이미 설치된 환경 기준
# # --extra-index-url 추가로 기존 torch를 건드리지 않고 설치
# # ──────────────────────────────────────────────────────────────
# import subprocess

# subprocess.run([
#     'pip', 'install', '-q',
#     'chandra-ocr[hf]',
#     'transformers', 'accelerate', 'pillow',
#     'fastapi', 'uvicorn', 'python-multipart',
#     '--extra-index-url', 'https://download.pytorch.org/whl/cu118'
# ])

In [3]:
# ──────────────────────────────────────────────────────────────
# 임포트 및 GPU 확인
# ──────────────────────────────────────────────────────────────
import torch
import base64
import threading
import io

from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

# GPU 사용 가능 여부 확인
deviceType = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'사용 디바이스: {deviceType}')

if deviceType == 'cuda':
    vramGB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {vramGB:.1f} GB')
else:
    print(' GPU를 찾을 수 없습니다. CPU로 실행되며 속도가 느릴 수 있습니다.')

c:\Users\SMT13\ai-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


사용 디바이스: cuda
GPU: NVIDIA RTX 2000 Ada Generation
VRAM: 16.0 GB


In [4]:
if torch.cuda.get_device_capability()[0] >= 8:
    # 고속 attention 메커니즘을 구현하는 라이브러리
    attn_implementation = "flash_attention_2"
    torch_dtype = torch.bfloat16
else:
    attn_implementation = "eager"
    torch_dtype = torch.float16


In [5]:
attn_implementation

'flash_attention_2'

In [6]:
# ──────────────────────────────────────────────────────────────
# 모델 로드 — 4bit NF4 양자화 적용
# ──────────────────────────────────────────────────────────────
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

MODEL_ID = 'datalab-to/chandra-ocr-2'

torchDtype = torch.bfloat16

quantConfig = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torchDtype,
    bnb_4bit_use_double_quant=False
)

print(f'모델 로딩 중: {MODEL_ID}')

model = AutoModelForImageTextToText.from_pretrained(
MODEL_ID,
    quantization_config=quantConfig,
    attn_implementation="sdpa",
    device_map='auto',
)
model.eval()
model.processor = AutoProcessor.from_pretrained(MODEL_ID)
model.processor.tokenizer.padding_side = 'left'

print('모델 로드 완료!')
if deviceType == 'cuda':
    usedVram = torch.cuda.memory_allocated(0) / (1024 ** 3)
    print(f'현재 VRAM 사용량: {usedVram:.2f} GB')

모델 로딩 중: datalab-to/chandra-ocr-2


W0423 19:07:48.847000 19168 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 724/724 [00:04<00:00, 172.08it/s]


모델 로드 완료!
현재 VRAM 사용량: 3.31 GB


In [7]:
# ──────────────────────────────────────────────────────────────
# OCR 추론 함수 — 공식 chandra-ocr 패키지 사용
# prompt_type: 'ocr' (빠름, 텍스트만) / 'ocr_layout' (느림, 마크다운 레이아웃)
# ──────────────────────────────────────────────────────────────
import io
from chandra.model.hf import generate_hf
from chandra.model.schema import BatchInputItem
from chandra.output import parse_markdown

MAX_IMAGE_SIZE = 1024  # 긴 변 기준 최대 픽셀

def resizeImage(pilImage: Image.Image) -> Image.Image:
    """ 이미지가 너무 크면 비율 유지하며 축소 """
    w, h = pilImage.size
    if max(w, h) > MAX_IMAGE_SIZE:
        ratio = MAX_IMAGE_SIZE / max(w, h)
        pilImage = pilImage.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
    return pilImage

def runOcr(pilImage: Image.Image, promptType: str = 'ocr') -> str:
    """ PIL 이미지를 받아 Chandra OCR 2로 텍스트를 추출하여 반환 """
    if pilImage.mode != 'RGB':
        pilImage = pilImage.convert('RGB')

    pilImage = resizeImage(pilImage)
    print(f'처리 이미지 크기: {pilImage.size}, 모드: {promptType}')

    batch = [
        BatchInputItem(
            image=pilImage,
            prompt_type=promptType
        )
    ]

    result = generate_hf(batch, model)[0]
    return parse_markdown(result.raw)


def bytesToPilImage(imageBytes: bytes) -> Image.Image:
    """ 바이트 데이터를 PIL Image 객체로 변환 """
    return Image.open(io.BytesIO(imageBytes))

In [8]:
# ──────────────────────────────────────────────────────────────
# 간단한 테스트 예시 (로컬 이미지 파일로 바로 테스트)
# ──────────────────────────────────────────────────────────────
import os

# 테스트할 이미지 경로 (본인 이미지로 변경)
testImagePath = 'data.jpg'

if os.path.exists(testImagePath):
    testImage = Image.open(testImagePath)
    print('OCR 실행 중...')
    extractedText = runOcr(testImage)
    print('=' * 50)
    print('추출된 텍스트:')
    print(extractedText)
    print('=' * 50)
else:
    print(f'테스트 이미지({testImagePath})가 없습니다.')
    print('같은 폴더에 test.jpg를 넣고 다시 실행하거나,')
    print('아래 API 서버를 실행해서 /ocr 엔드포인트로 테스트하세요.')

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


OCR 실행 중...
처리 이미지 크기: (484, 1024), 모드: ocr


c:\Users\SMT13\ai-project\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


추출된 텍스트:
# 실제 서버 설치 시 핵심 주의사항 및 네트워크 설정 가이드

1

## 초기 부팅 설정 (BIOS/UEFI)

![A computer monitor displaying a BIOS screen with various settings and a mouse cursor.](eddffb734c4610969a8b2bc8e60b4a24_4_img.webp)

- • Rufus 같은 툴로 만든 설치 USB 연결
- • PC 전원 ON 시 F2/DEL 키로 BIOS 진입
- • 'Boot' 메뉴에서 USB 장치를 1순위로 설정 후 저장

2

## 활성 네트워크 인터페이스 찾기

![A hand holding a blue cable and plugging it into a server rack with multiple network ports.](eddffb734c4610969a8b2bc8e60b4a24_8_img.webp)

🔍 트러블슈팅 절차:

- • 만약 원하는 인터페이스가 'DOWN' 상태라면:
  - • [기본 필수 확인] 모든 인터페이스 상태 조회:  
    ip link show 명령어로 상태 및 케이블 연결(LOWER\_UP) 여부 재확인
  - • 전부 강제 작동 (또는 특정 인터페이스):  
    sudo ip link set <인터페이스명> up  
    sudo ip link set dev <인터페이스명> up
  - • UP 후 살아있는지 확인 (재검증)

3

## 네트워크 영구 설정 (Ubuntu vs. RedHat)

![A comparison graphic showing the Ubuntu logo (orange circle with a white 'U') on the left and the Red Hat logo (red fedora hat) on the right, separated by the text 'VS'.](eddffb734c4610969a8b2bc8e60b4a24_13_img.webp)

- • **R

In [ ]:
# ──────────────────────────────────────────────────────────────
# FastAPI 서빙 서버 (포트 8001)
# ──────────────────────────────────────────────────────────────
import uvicorn
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse

ocrApp = FastAPI(title='Chandra OCR 2 서버')

ocrApp.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)


@ocrApp.post('/ocr')
async def extractText(image: UploadFile = File(...)):
    """ 이미지를 받아 Chandra OCR 2로 텍스트를 추출하는 API (포트 8001) """
    try:
        allowedTypes = ['image/jpeg', 'image/png', 'image/webp', 'image/gif', 'image/bmp', 'image/tiff']
        if image.content_type not in allowedTypes:
            raise HTTPException(status_code=400, detail='지원하지 않는 이미지 형식입니다.')

        imageBytes = await image.read()
        pilImage   = bytesToPilImage(imageBytes)
        ocrResult  = runOcr(pilImage)

        return JSONResponse(content={
            'success':       True,
            'model':         MODEL_ID,
            'extracted_text': ocrResult
        })

    except HTTPException as httpErr:
        raise httpErr
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={'success': False, 'message': str(e)}
        )


@ocrApp.get('/')
async def healthCheck():
    """ OCR 서버 상태 확인 """
    return {'success': True, 'model': MODEL_ID, 'device': deviceType}


# 백그라운드 스레드로 서버 실행
def runOcrServer():
    """ uvicorn OCR 서버를 백그라운드 스레드에서 실행 """
    uvicorn.run(ocrApp, host='0.0.0.0', port=8001)

ocrThread = threading.Thread(target=runOcrServer, daemon=True)
ocrThread.start()

print('OCR 서버 시작: http://localhost:8001')
print('API 문서:     http://localhost:8001/docs')
print()
print('테스트 명령어:')
print('  curl -X POST http://localhost:8001/ocr -F "image=@test.jpg"')

OCR 서버 시작: http://localhost:8001
API 문서:     http://localhost:8001/docs

테스트 명령어:
  curl -X POST http://localhost:8001/ocr -F "image=@test.jpg"


INFO:     Started server process [19168]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


## 포트 구성

| 서버 | 포트 | 역할 |
|------|------|------|
| analysis_server.ipynb | 8000 | OLLAMA/GPT 이미지 분석 |
| **analysis_ocr.ipynb** | **8001** | **Chandra OCR 2 텍스트 추출** |
| simple_web | 3000 | 웹 UI |

## 트러블슈팅

```
CUDA out of memory  →  model.generate() 의 max_new_tokens를 512로 줄이거나
                        torch_dtype=torch.float16 확인

모델 로드 실패       →  pip install -U transformers accelerate 재실행
```